# Exercise 1: image → JSON → DataFrame

## Tasks

1. **One image:** send a single picture to the OpenAI API and read its description.
2. **Structure:** ask for a fixed JSON format with a few fields.
3. **More images:** repeat for a small folder and collect the answers in a table.
4. **Evaluate:** compare the table with hand labels, field by field.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret.

This notebook uses the **OpenAI API** (model `gpt-6-luna`). Other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code. The API calls are in `helper.py` – open it to see how they work.

In [ ]:
# Standard Python libraries.
import os
import sys
from pathlib import Path
from typing import Literal

# Libraries installed from requirements.txt.
import jiwer
import pandas as pd
from pydantic import BaseModel
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer

# Work inside this exercise folder, wherever Jupyter starts.
if not Path("data").exists():  # we are inside an exercise folder
    os.chdir("..")  # go to the main folder
os.chdir("01_Image_Extraction")
sys.path.append("..")  # helper.py is in the main folder

In [ ]:
# Shared helpers from helper.py.
from helper import DATA, ask, describe, image_to_input, show_images

## 1. Send one image to the OpenAI API and read its description

Pick one image. Other images in `data/images/`: `street.jpg`, `sheep.jpg`, `parking.jpg`, `lake_night.jpg`, `lake_swan.jpg`.

In [ ]:
# The image to describe. Change the filename to try another one.
image_path = DATA / "images" / "lake.jpg"
show_images([image_path])

The OpenAI API request is text, so the image bytes are converted to text (base64):

In [ ]:
# The start of the image as text.
image_input = image_to_input(image_path)
print(image_input["image_url"][:80], "...")

In [ ]:
# Send a question and the image to the OpenAI API.
print(describe("Describe this image in two sentences.", image_path))

The image is read as tokens, so every image costs money. Every call is written to `api_log.jsonl` – its last line holds the tokens of the call above. Estimate before you run: cost per image × number of images.

In [ ]:
# OpenAI prices for gpt-6-luna.
PRICE_INPUT = 0.10   # US$ per 1 million input tokens
PRICE_OUTPUT = 0.50  # US$ per 1 million output tokens

# Tokens of the last call, from the log file.
log = pd.read_json("api_log.jsonl", lines=True)
last = log.iloc[-1]
print("Input tokens: ", last["input_tokens"])
print("Output tokens:", last["output_tokens"])

# Cost of this call and of a folder with 1,000 images.
input_cost = last["input_tokens"] * PRICE_INPUT / 1_000_000
output_cost = last["output_tokens"] * PRICE_OUTPUT / 1_000_000
cost = input_cost + output_cost
print(f"This image:   US$ {cost:.5f}")
print(f"1,000 images: US$ {cost * 1000:.2f}")

## 2. Ask for a fixed JSON format

A free text is hard to compare. Now the answer must contain exactly these fields, written as a Python class. `None` means no text is visible.

In [ ]:
# The answer format: one field per piece of information.
class ImageRecord(BaseModel):
    scene: Literal["lake", "parking lot", "alpine pasture", "city street"]
    objects: list[str]
    people_count: int
    text_in_image: str | None


# The instruction for the model.
PROMPT = (
    "Describe this image. Pick the scene from the list. "
    "Name up to five prominent objects as lowercase singular nouns. "
    "Count the visible people. "
    "Copy the most prominent text. Use null if there is no text "
    "and [unreadable] if the text cannot be read."
)

In [ ]:
# Same image, now with a fixed answer format.
answer = ask(PROMPT, ImageRecord, [image_path])
answer

## 3. Repeat for all images and collect the answers in a table

In [ ]:
# All images in the course folder.
folder = DATA / "images"
paths = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png"))
print(len(paths), "images")

In [ ]:
# Ask the model about every image and add the filename ourselves.
rows = []
for path in paths:
    row = ask(PROMPT, ImageRecord, [path])
    row["file"] = path.name
    rows.append(row)

# One row per image; save the table as JSON lines.
extracted = pd.DataFrame(rows)
extracted.to_json("extracted.jsonl", orient="records", lines=True)
extracted

## 4. Compare the table with the hand labels, field by field

The average hides the weak field – so each field gets its own score.

In [ ]:
# Hand labels: one row per image. Objects are separated by ";".
labels = pd.read_csv(folder / "labels.csv", keep_default_na=False)

# Put prediction and label side by side, matched by filename.
scores = extracted.merge(labels, on="file", suffixes=("_pred", "_ref"))

# Exact match: share of images where model and label agree.
scene = accuracy_score(scores["scene_ref"], scores["scene_pred"])
people = accuracy_score(
    scores["people_count_ref"], scores["people_count_pred"]
)
print("Scene accuracy:       ", round(scene, 2))
print("People count accuracy:", round(people, 2))

**Objects** are a list, so an exact match is too strict. F1 combines
- *precision*: how many of the named objects are correct?
- *recall*: how many of the true objects were found?

In [ ]:
# Object lists from the model and from the labels.
pred = scores["objects_pred"]
ref = scores["objects_ref"].str.split(";")  # "boat;water" -> ["boat", "water"]

# Turn each list into a row of 0s and 1s, one column per object name.
encoder = MultiLabelBinarizer()
encoder.fit(pd.concat([pred, ref]))
y_pred = encoder.transform(pred)
y_ref = encoder.transform(ref)

# Averages over all images.
precision = precision_score(y_ref, y_pred, average="samples", zero_division=0)
recall = recall_score(y_ref, y_pred, average="samples", zero_division=0)
f1 = f1_score(y_ref, y_pred, average="samples", zero_division=0)
print("Object precision:", round(precision, 2))
print("Object recall:   ", round(recall, 2))
print("Object F1:       ", round(f1, 2))

**Text** is scored with the character error rate (CER): how many characters must be changed, per character of the label? Lower is better.

In [ ]:
# Keep only images whose label contains readable text.
with_text = scores[~scores["text_in_image_ref"].isin(["null", "[unreadable]"])]

# No text from the model counts as empty text.
reference = list(with_text["text_in_image_ref"])
prediction = list(with_text["text_in_image_pred"].fillna(""))
print("Model:", prediction, "| Label:", reference)
print("CER:", round(jiwer.cer(reference, prediction), 2))

**Check yourself:** Why can a perfectly valid JSON answer still be wrong? Compare the two object lists of a few images.

In [ ]:
# Model and label objects side by side.
scores[["file", "objects_pred", "objects_ref"]]

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')